# 03 · Refinement with CUPED: the same experiment, a sharper estimate

> **Role of this notebook: refinement of the analysis of X5's data.**
> The experiment was run by X5 Retail Group in 2019, not by me. Notebook `02` analysed its published results;
> this notebook re-estimates the *same* effect on the *same* customers, using each customer's purchase history
> from *before* the SMS (CUPED) to remove noise. Nothing new is launched and no new data about the campaign is created.
>
> Order of the project: **01 design** → *execution by X5, not part of this project* → **02 analysis** →
> **03 refinement** (this notebook) → recommendation.

## 1. What CUPED is, in plain words

Customers differ a lot *before* any SMS is sent: some buy every week, others almost never. That spread makes the
purchase rate noisy, so a small effect is hard to see. **CUPED** (Controlled-experiment Using Pre-Experiment Data)
subtracts from each customer's outcome the part that could already be predicted from their past behaviour:

`y_adjusted = y − θ · (x − mean(x))`, with `θ = cov(x, y) / var(x)`

where `x` is a pre-campaign covariate (here: past spend and number of past transactions) and `y` is the outcome (`target`, purchased or not).

**Why it does not bias the effect.** `x` is measured *before* the SMS, so the SMS cannot have changed it, and
randomisation makes it look the same in both arms *on average*. The adjustment therefore shifts treated and control customers by the
same amount in expectation, so the *difference* between them is not biased; the noise around it shrinks.
In one real sample, though, the two arms are never perfectly identical in `x`, and CUPED also corrects that small chance imbalance:
the estimate can move slightly (by exactly `−θ ×` the difference in the covariate between arms). I check the balance below and measure that shift in section 5.

**How much it helps** depends on ρ, the correlation between past and future behaviour. The variance of the outcome
shrinks by a factor `1 − ρ²`, so the confidence interval gets narrower by a factor `√(1 − ρ²)`. With a low ρ the gain is small.

## 2. Data

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

sys.path.insert(0, str(Path("..") / "src"))
from ab_utils import attach_history, cuped_adjust, diff_in_means, n_per_arm, n_with_cuped

DATA = Path("..") / "data_raw"
HOW_TO_GET = {
    "clients.csv.gz": "Download it from the Hugging Face dataset `pytorch-lifestream/retailhero-uplift` into `data_raw/` (see README).",
    "uplift_train.csv.gz": "Download it from the Hugging Face dataset `pytorch-lifestream/retailhero-uplift` into `data_raw/` (see README).",
    "purchases_features.csv.gz": "Download `purchases.csv.gz` from the same dataset into `data_raw/` and run `scripts/build_purchase_features.py` (see README).",
}
for name, hint in HOW_TO_GET.items():
    if not (DATA / name).exists():
        raise FileNotFoundError(f"{DATA / name} not found. {hint}")

clients = pd.read_csv(DATA / "clients.csv.gz", parse_dates=["first_issue_date", "first_redeem_date"])
train = pd.read_csv(DATA / "uplift_train.csv.gz")
features = pd.read_csv(DATA / "purchases_features.csv.gz", parse_dates=["last_purchase_date"])

# Same merge as notebook 02, then attach the pre-campaign purchase history WITHOUT dropping anyone
df = train.merge(clients, on="client_id", how="left", validate="1:1")
df = attach_history(df, features)
assert len(df) == len(train), "attaching the history changed the number of customers"

print(f"Customers: {len(df):,}  (same as uplift_train: {len(train):,})")
print(f"With purchase history: {df.has_history.mean():.2%}  |  without: {(~df.has_history).sum():,}")

Customers: 200,039  (same as uplift_train: 200,039)
With purchase history: 100.00%  |  without: 0


## 3. Is the covariate really "pre-campaign"?

CUPED is only valid if the covariate was measured before the treatment. Two pieces of evidence, stated honestly:

1. **The source's claim.** The dataset description says the purchase file is history *before* the SMS campaign. I take that as the source's word, not as something I verified independently.
2. **My own date check.** The real campaign date is not published. As a proxy for the snapshot I use the latest card issue date in `clients` (a lower bound). If purchases kept arriving after the campaign, customers who received the SMS could buy *more* afterwards and the covariate would be contaminated by the treatment. So I compare the share of customers with a purchase dated after the proxy date in each arm: if the SMS leaked into the file, it should differ.

In [2]:
proxy_snapshot = clients.first_issue_date.max()
cutoff = proxy_snapshot.normalize() + pd.Timedelta(days=1)
print(f"Latest card issue date (snapshot proxy): {proxy_snapshot:%Y-%m-%d}")
print(f"Latest purchase in the history file:     {df.last_purchase_date.max():%Y-%m-%d}")

late = (df.last_purchase_date >= cutoff)  # missing dates compare as False
share = late.groupby(df.treatment_flg).mean()
chi2, p_late, _, _ = stats.chi2_contingency(pd.crosstab(df.treatment_flg, late), correction=False)
print(f"Share of customers with a purchase on or after {cutoff:%Y-%m-%d}: "
      f"control {share[0]:.2%} vs treated {share[1]:.2%}  (chi-square p = {p_late:.2f})")

Latest card issue date (snapshot proxy): 2019-03-15
Latest purchase in the history file:     2019-03-18
Share of customers with a purchase on or after 2019-03-16: control 38.02% vs treated 38.03%  (chi-square p = 0.95)


If the SMS had leaked into the purchase history, the treated arm would show a different share. A similar share in both arms (large p-value) is consistent with a clean history, but it is evidence, not proof: the real campaign date remains unknown.

## 4. Validity assertions

Before adjusting anything, two conditions must hold. The notebook **stops** if either fails:

- **Balance:** the covariate must look the same in both arms, |SMD| < 0.1 (standardised mean difference, the same rule used in notebook `02`).
- **Stable θ:** past spend is heavy-tailed (a few customers spend far more than the rest), so I use `log1p(spend)` and the number of transactions instead of raw sums, and check that θ is a finite number.

In [3]:
y = df.target.to_numpy()
t = df.treatment_flg.to_numpy()
COVARIATES = {"log1p(total spend)": "log_spend", "Number of transactions": "n_transactions"}


def smd(x, t):
    """Standardised mean difference, treated minus control (same formula as notebook 02)."""
    a, b = x[t == 1], x[t == 0]
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)


rows = []
for label, col in COVARIATES.items():
    x = df[col].to_numpy(dtype=float)
    s = smd(x, t)
    _, theta = cuped_adjust(y, x)
    assert abs(s) < 0.1, f"{label}: |SMD| = {abs(s):.3f} >= 0.1, the covariate is not balanced between arms"
    assert np.isfinite(theta), f"{label}: theta is not finite"
    rows.append({"Covariate": label, "SMD (treated - control)": s, "rho with target": np.corrcoef(x, y)[0, 1], "theta": theta})

display(pd.DataFrame(rows).set_index("Covariate").round(3))
print("Both covariates are balanced between arms (|SMD| < 0.1) and theta is finite.")

,SMD (treated - control),rho with target,theta
Covariate,,,
log1p(total spend),0.010,0.324,0.156
Number of transactions,0.007,0.357,0.010


Both covariates are balanced between arms (|SMD| < 0.1) and theta is finite.


## 5. Estimate: original vs CUPED

`diff_in_means` gives the treated-minus-control difference with an unpooled 95 % confidence interval.
First on the raw outcome (this must reproduce notebook `02`), then on the CUPED-adjusted outcome.

The expected shrinkage of the interval is `√(1 − ρ²)`. I accept the measured width ratio if it is within **0.01** of that theoretical value.
This tolerance was fixed *before* looking at the result: it is about one sixth of the typical gain, so it would catch a CUPED that does nothing, and it still leaves room for the small
difference between a theory that uses one coefficient for everyone and an interval that uses each arm's own variance.

In [4]:
WIDTH_RATIO_TOLERANCE = 0.01  # fixed in advance (see text above)

eff0, lo0, hi0 = diff_in_means(y, t)
width0 = hi0 - lo0
print(f"Original: effect {eff0*100:+.2f} pp, 95% CI {lo0*100:+.2f} to {hi0*100:+.2f} pp (width {width0*100:.2f} pp)")

results = [{"Estimate": "Original (no adjustment)", "rho": np.nan, "Effect (pp)": eff0 * 100,
            "CI low (pp)": lo0 * 100, "CI high (pp)": hi0 * 100, "CI width (pp)": width0 * 100,
            "Width vs original": 1.0, "Theory sqrt(1-rho^2)": np.nan}]

for label, col in COVARIATES.items():
    x = df[col].to_numpy(dtype=float)
    rho = np.corrcoef(x, y)[0, 1]
    y_adj, theta = cuped_adjust(y, x)
    eff, lo, hi = diff_in_means(y_adj, t)
    ratio, theory = (hi - lo) / width0, np.sqrt(1 - rho**2)

    assert (hi - lo) <= width0, f"{label}: the adjusted interval is wider than the original"
    assert lo0 <= eff <= hi0, f"{label}: the adjusted estimate left the original interval"
    assert abs(ratio - theory) < WIDTH_RATIO_TOLERANCE, (
        f"{label}: width ratio {ratio:.4f} differs from sqrt(1 - rho^2) = {theory:.4f} by more than {WIDTH_RATIO_TOLERANCE}")

    results.append({"Estimate": f"CUPED with {label}", "rho": rho, "Effect (pp)": eff * 100,
                    "CI low (pp)": lo * 100, "CI high (pp)": hi * 100, "CI width (pp)": (hi - lo) * 100,
                    "Width vs original": ratio, "Theory sqrt(1-rho^2)": theory})

table = pd.DataFrame(results).set_index("Estimate")
display(table.round(3))
print("All assertions passed.")

Original: effect +3.32 pp, 95% CI +2.90 to +3.75 pp (width 0.85 pp)


,rho,Effect (pp),CI low (pp),CI high (pp),CI width (pp),Width vs original,Theory sqrt(1-rho^2)
Estimate,,,,,,,
Original (no adjustment),NaN,3.323,2.898,3.748,0.850,1.000,NaN
CUPED with log1p(total spend),0.324,3.174,2.771,3.576,0.805,0.946,0.946
CUPED with Number of transactions,0.357,3.197,2.799,3.594,0.794,0.934,0.934


All assertions passed.


The point estimate also moves slightly (see the "Effect" column). That is not a bug: with CUPED the shift is exactly `−θ × (mean of x in treated − mean of x in control)`. The next cell checks that identity.

In [5]:
x = df["log_spend"].to_numpy(dtype=float)
_, theta = cuped_adjust(y, x)
gap = x[t == 1].mean() - x[t == 0].mean()
predicted_shift = -theta * gap
measured_shift = results[1]["Effect (pp)"] / 100 - eff0
print(f"Treated customers had a mean log1p(spend) {gap:+.4f} higher than control by chance (SMD {smd(x, t):+.3f}).")
print(f"Predicted shift -theta * gap = {predicted_shift*100:+.3f} pp  |  measured shift = {measured_shift*100:+.3f} pp")
assert np.isclose(predicted_shift, measured_shift), "the shift is not explained by the covariate imbalance"

Treated customers had a mean log1p(spend) +0.0096 higher than control by chance (SMD +0.010).
Predicted shift -theta * gap = -0.149 pp  |  measured shift = -0.149 pp


## 6. How much did it help, and why

The first row matches notebook `02`: it is the same estimate on the same customers. With CUPED, the confidence interval gets narrower by the factor the theory predicts for each covariate (about 5-7 %).
The gain is limited by ρ: past behaviour predicts future behaviour only moderately here (ρ is well below 0.5), so the interval shrinks by a few percent, not by half.

**The point estimate also moves, and it is worth reading carefully.** It goes from +3.32 pp to about +3.2 pp, still well inside the original interval. By chance, the treated customers had slightly higher past spend than the control customers,
so part of the raw difference reflected that head start; CUPED removes it (the cell above shows the shift is exactly the imbalance times θ). So the more careful summary is *about +3.2 pp with a slightly narrower interval*, rather than +3.3 pp.
The conclusion does not change: the SMS raises purchases. What CUPED gives here is a small gain in precision and a small correction, not a different story.

## 7. Back to the design in notebook 01

Notebook `01` proposed a design for the *next* test and showed what CUPED could save for illustrative ρ values of 0.3, 0.5 and 0.7, "not estimated from data".
Here is the real ρ from X5's data, applied to the same design (minimum detectable effect of 1 pp).

In [6]:
control = df.loc[df.treatment_flg == 0, "target"]
p = control.mean()                       # baseline purchase rate, as in notebook 01
n_plain = n_per_arm(p, 0.01)             # chosen MDE of notebook 01: 1 pp
rho_real = abs(table.loc["CUPED with log1p(total spend)", "rho"])
n_adj = n_with_cuped(n_plain, rho_real)

print(f"Baseline p = {p:.3f}  |  design of notebook 01 (MDE 1 pp): {n_plain:,} customers per arm without CUPED")
print(f"Real rho (log1p spend vs purchase) = {rho_real:.2f}  ->  with CUPED: {n_adj:,} per arm "
      f"({(1 - n_adj / n_plain):.1%} fewer customers)")

illustrative = pd.DataFrame({
    "rho": [0.3, 0.5, 0.7, rho_real],
    "Where it comes from": ["illustrative (notebook 01)", "illustrative (notebook 01)", "illustrative (notebook 01)", "estimated from X5's data"],
})
illustrative["Customers per arm"] = [n_with_cuped(n_plain, r) for r in illustrative.rho]
illustrative["Saved vs no CUPED"] = (1 - illustrative["Customers per arm"] / n_plain).map("{:.1%}".format)
display(illustrative.round({"rho": 3}).set_index("rho"))

Baseline p = 0.603  |  design of notebook 01 (MDE 1 pp): 37,570 customers per arm without CUPED
Real rho (log1p spend vs purchase) = 0.32  ->  with CUPED: 33,627 per arm (10.5% fewer customers)


,Where it comes from,Customers per arm,Saved vs no CUPED
rho,,,
0.300,illustrative (notebook 01),34189,9.0%
0.500,illustrative (notebook 01),28178,25.0%
0.700,illustrative (notebook 01),19161,49.0%
0.324,estimated from X5's data,33627,10.5%


**Reading it.** The real ρ falls at the low end of the illustrative range: only a little above the 0.3 case and far from the optimistic 0.7. For a next test with this customer base I would therefore plan with a modest saving from CUPED
(roughly a tenth of the sample), and treat anything larger as a bonus rather than a promise.